# Notebook for clustering.

Run interactive notebooks in the following order:\
preprocessing.ipynb  ->  plot_corr_assoc.ipynb  ->  dimensionality_reduction.ipynb  ->  clustering.ipynb

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import (  # type: ignore
    DBSCAN,
    HDBSCAN,
    OPTICS,
    AffinityPropagation,
    AgglomerativeClustering,
    KMeans,
)
from sklearn.metrics import adjusted_rand_score, rand_score
from sklearn.mixture import BayesianGaussianMixture
from umap import UMAP

from utils.clustering_utils import plot_2d_data, plot_silhouette, print_clustering_info, reduce_dimen_2d
from utils.utils import load_data, save_data, save_plot

pd.options.mode.copy_on_write = True

In [ ]:
# Create RNG for reproducibility
rng = np.random.default_rng(394325)

# Hardcoded paths, change to apply to new data
data_path = Path("outputs/dim_red/esa_2018_2021_reduced.pkl")
vars_info_path = Path("data/preprocessed/esa_vars_info.pkl")
outputs_folder = "outputs/clustering"

# Read in data
data: pd.DataFrame = load_data(data_path)
vars_info: pd.DataFrame = load_data(vars_info_path)

# # Normalize data
# from sklearn.preprocessing import StandardScaler
# standard_scaler = StandardScaler()
# data_transformed = standard_scaler.fit_transform(data)

# Create distance matrix
# from scipy.spatial.distance import pdist, squareform
# distances = pdist(data.values, metric='euclidean')
# dist_matrix = squareform(distances)
# ...

In [ ]:
# Plot projected data in 2d
rng = np.random.default_rng(23523)

projection_method = "TSNE"  # projection_method="UMAP"
tsne_kwargs = dict(perplexity=100, early_exaggeration=20)
data_2d = reduce_dimen_2d(data=data, projection_method=projection_method, rng=rng, **tsne_kwargs)

data_plot = plot_2d_data(data=data_2d)
data_plot.show()


# Save plot
data_plot_save_path = Path(outputs_folder, f"data_plot_{projection_method}.png")
save_plot(plot=data_plot, save_path=data_plot_save_path, label=f"plot of data projected by {projection_method}")

# Show plot
data_plot.show()

### k-means clustering with silhouette plots

In [ ]:
# rng = np.random.default_rng(5252)

# kmeans_n_clusters_list = [4, 5, 6, 7]
# kmeans_clustering_list = []
# kmeans_cluster_labels_list = []
# kmeans_silhouette_plot_list = []
# for n_clusters in kmeans_n_clusters_list:
#     clustering = KMeans(n_clusters=n_clusters, random_state=np.random.RandomState(rng.bit_generator))
#     cluster_labels = pd.Series(clustering.fit_predict(data)).replace(-1, pd.NA).astype("category")

#     # Print information about clustering
#     print_clustering_info(cluster_labels)

#     # Create silhouette plot
#     silhouette_plot = plot_silhouette(data, cluster_labels)

#     # Append to list
#     kmeans_clustering_list.append(clustering)
#     kmeans_cluster_labels_list.append(cluster_labels)
#     kmeans_silhouette_plot_list.append(silhouette_plot)

# p1, p2, p3, p4 = kmeans_silhouette_plot_list
# kmeans_silhouette_plots = (p1 | p2) / (p3 | p4)
# # Save plot
# kmeans_silhouette_plots_save_path = Path(outputs_folder, "kmeans_silhouette_plots.png")
# save_plot(plot=kmeans_silhouette_plots, save_path=kmeans_silhouette_plots_save_path, label="k-means silhouette plots")
# # Show plot
# kmeans_silhouette_plots.draw()

In [ ]:
rng = np.random.default_rng(5252)

kmeans = KMeans(n_clusters=10, random_state=np.random.RandomState(rng.bit_generator))
kmeans_labels = pd.Series(kmeans.fit_predict(data)).replace(-1, pd.NA).astype("category")

# Print information about clustering
print_clustering_info(kmeans_labels)

In [ ]:
# Silhouette plot
kmeans_silhouette_plot = plot_silhouette(data, kmeans_labels)

# Save plot
kmeans_silhouette_plot_save_path = Path(outputs_folder, "kmeans_silhouette_plot.png")
save_plot(plot=kmeans_silhouette_plot, save_path=kmeans_silhouette_plot_save_path, label="k-means silhouette plot")

# Show plot
kmeans_silhouette_plot.show()

In [ ]:
# Plot clustering
kmeans_plot = plot_2d_data(data=data_2d, cluster_labels=kmeans_labels)

# Save plot
kmeans_plot_save_path = Path(outputs_folder, "kmeans_plot.png")
save_plot(plot=kmeans_plot, save_path=kmeans_plot_save_path, label="plot of k-means clustering")

# Show plot
kmeans_plot.show()

### Affinity propagation clustering

In [ ]:
rng = np.random.default_rng(34522)

ap = AffinityPropagation(damping=0.5, preference=None, random_state=np.random.RandomState(rng.bit_generator))
ap_labels = pd.Series(ap.fit_predict(data)).replace(-1, pd.NA).astype("category")

# Print information about clustering
print_clustering_info(ap_labels)

-> crashes

### Agglomerative clustering

Ward linkage

In [ ]:
ward_linkage = AgglomerativeClustering(n_clusters=10, linkage="ward")
ward_linkage_labels = pd.Series(ward_linkage.fit_predict(data)).replace(-1, pd.NA).astype("category")

# Print information about clustering
print_clustering_info(ward_linkage_labels)

In [ ]:
# Silhouette plot
ward_linkage_silhouette_plot = plot_silhouette(data, ward_linkage_labels)

# Save plot
ward_linkage_silhouette_plot_save_path = Path(outputs_folder, "ward_linkage_silhouette_plot.png")
save_plot(
    plot=ward_linkage_silhouette_plot,
    save_path=ward_linkage_silhouette_plot_save_path,
    label="Ward linkage silhouette plot",
)

# Show plot
ward_linkage_silhouette_plot.show()

In [ ]:
# Plot clustering
ward_linkage_plot = plot_2d_data(data=data_2d, cluster_labels=ward_linkage_labels)

# Save plot
ward_linkage_plot_save_path = Path(outputs_folder, "ward_linkage_plot.png")
save_plot(plot=ward_linkage_plot, save_path=ward_linkage_plot_save_path, label="plot of Ward linkage clustering")

# Show plot
ward_linkage_plot.show()

Average linkage

In [ ]:
average_linkage = AgglomerativeClustering(n_clusters=10, linkage="average")
average_linkage_labels = pd.Series(average_linkage.fit_predict(data)).replace(-1, pd.NA).astype("category")

# Print information about clustering
print_clustering_info(average_linkage_labels)

In [ ]:
# Silhouette plot
average_linkage_silhouette_plot = plot_silhouette(data, average_linkage_labels)

# Save plot
average_linkage_silhouette_plot_save_path = Path(outputs_folder, "average_linkage_silhouette_plot.png")
save_plot(
    plot=average_linkage_silhouette_plot,
    save_path=average_linkage_silhouette_plot_save_path,
    label="average linkage silhouette plot",
)

# Show plot
average_linkage_silhouette_plot.show()

In [ ]:
# Plot clustering
average_linkage_plot = plot_2d_data(data=data_2d, cluster_labels=average_linkage_labels)

# Save plot
average_linkage_plot_save_path = Path(outputs_folder, "average_linkage_plot.png")
save_plot(
    plot=average_linkage_plot, save_path=average_linkage_plot_save_path, label="plot of average linkage clustering"
)

# Show plot
average_linkage_plot.show()

Single linkage

In [ ]:
single_linkage = AgglomerativeClustering(n_clusters=10, linkage="single")
single_linkage_labels = pd.Series(single_linkage.fit_predict(data)).replace(-1, pd.NA).astype("category")

# Print information about clustering
print_clustering_info(single_linkage_labels)

In [ ]:
# Silhouette plot
single_linkage_silhouette_plot = plot_silhouette(data, single_linkage_labels)

# Save plot
single_linkage_silhouette_plot_save_path = Path(outputs_folder, "single_linkage_silhouette_plot.png")
save_plot(
    plot=single_linkage_silhouette_plot,
    save_path=single_linkage_silhouette_plot_save_path,
    label="single linkage silhouette plot",
)

# Show plot
single_linkage_silhouette_plot.show()

In [ ]:
# Plot clustering
single_linkage_plot = plot_2d_data(data=data_2d, cluster_labels=single_linkage_labels)

# Save plot
single_linkage_plot_save_path = Path(outputs_folder, "single_linkage_plot.png")
save_plot(plot=single_linkage_plot, save_path=single_linkage_plot_save_path, label="plot of single linkage clustering")

# Show plot
single_linkage_plot.show()

Complete linkage

In [ ]:
complete_linkage = AgglomerativeClustering(n_clusters=10, linkage="complete")
complete_linkage_labels = pd.Series(complete_linkage.fit_predict(data)).replace(-1, pd.NA).astype("category")

# Print information about clustering
print_clustering_info(complete_linkage_labels)

In [ ]:
# Silhouette plot
complete_linkage_silhouette_plot = plot_silhouette(data, complete_linkage_labels)

# Save plot
complete_linkage_silhouette_plot_save_path = Path(outputs_folder, "complete_linkage_silhouette_plot.png")
save_plot(
    plot=complete_linkage_silhouette_plot,
    save_path=complete_linkage_silhouette_plot_save_path,
    label="complete linkage silhouette plot",
)

# Show plot
complete_linkage_silhouette_plot.show()

In [ ]:
# Plot clustering
complete_linkage_plot = plot_2d_data(data=data_2d, cluster_labels=complete_linkage_labels)

# Save plot
complete_linkage_plot_save_path = Path(outputs_folder, "complete_linkage_plot.png")
save_plot(
    plot=complete_linkage_plot, save_path=complete_linkage_plot_save_path, label="plot of complete linkage clustering"
)

# Show plot
complete_linkage_plot.show()

### DBSCAN, OPTICS, and HDBSCAN clustering

In [ ]:
rng = np.random.default_rng(9458)

# dbscan = DBSCAN(eps=6.5, min_samples=10, n_jobs=-1)
# dbscan = OPTICS(min_samples=5, n_jobs=-1)
dbscan = HDBSCAN(
    min_cluster_size=10,
    min_samples=1,
    # cluster_selection_epsilon=1.0,
    store_centers=None,
    n_jobs=-1,
)
dbscan_labels = pd.Series(dbscan.fit_predict(data)).replace(-1, pd.NA).astype("category")
# dbscan_probs = dbscan.probabilities_  # HDBSCAN, ...
# dbscan_centroids = dbscan.centroids_  # HDBSCAN with store_centers="both"
# dbscan_medoids = dbscan.medoids_  # HDBSCAN with store_centers="both"

# Print information about clustering
print_clustering_info(dbscan_labels)

In [ ]:
# Plot clustering
dbscan_plot = plot_2d_data(data=data_2d, cluster_labels=dbscan_labels)

# Save plot
dbscan_plot_save_path = Path(outputs_folder, "dbscan_plot.png")
save_plot(plot=dbscan_plot, save_path=dbscan_plot_save_path, label="plot of HDBSCAN clustering")

# Show plot
dbscan_plot.show()

### Gaussian mixture model clustering with Dirichlet (process) prior

In [ ]:
rng = np.random.default_rng(2352)

gmm = BayesianGaussianMixture(
    n_components=10,
    covariance_type="full",
    weight_concentration_prior_type="dirichlet_process",
    weight_concentration_prior=1e3,
    mean_precision_prior=1.0,
    random_state=np.random.RandomState(rng.bit_generator),
)
gmm_labels = pd.Series(gmm.fit_predict(data)).replace(-1, pd.NA).astype("category")

# Print information about clustering
print_clustering_info(gmm_labels)

In [ ]:
# Silhouette plot
gmm_silhouette_plot = plot_silhouette(data, gmm_labels)

# Save plot
gmm_silhouette_plot_save_path = Path(outputs_folder, "gmm_silhouette_plot.png")
save_plot(plot=gmm_silhouette_plot, save_path=gmm_silhouette_plot_save_path, label="GMM silhouette plot")

# Show plot
gmm_silhouette_plot.show()

In [ ]:
# Plot clustering
gmm_plot = plot_2d_data(data=data_2d, cluster_labels=gmm_labels)

# Save plot
gmm_plot_save_path = Path(outputs_folder, "gmm_plot.png")
save_plot(plot=gmm_plot, save_path=gmm_plot_save_path, label="plot of GMM clustering")

# Show plot
gmm_plot.show()

### Compare clustering results

In [ ]:
# (Adjusted) Rand score

# labels0, labels1 = kmeans_labels, gmm_labels

labels0, labels1 = kmeans_labels, ward_linkage_labels

# labels0, labels1 = gmm_labels, ward_linkage_labels

# na_mask = dbscan_labels.isnull()
# labels0, labels1 = kmeans_labels[~ na_mask], dbscan_labels[~ na_mask]

# na_mask = dbscan_labels.isnull()
# labels0, labels1 = gmm_labels[~ na_mask], dbscan_labels[~ na_mask]

# na_mask = dbscan_labels.isnull()
# labels0, labels1 = ward_linkage_labels[~ na_mask], dbscan_labels[~ na_mask]

# rand_score(labels0, labels1)
adjusted_rand_score(labels0, labels1)

In [ ]:
# Table of clusterings

# labels0, labels1 = kmeans_labels, gmm_labels

labels0, labels1 = kmeans_labels, ward_linkage_labels

# labels0, labels1 = gmm_labels, ward_linkage_labels

# labels0, labels1 = kmeans_labels, dbscan_labels

# labels0, labels1 = gmm_labels, dbscan_labels

# labels0, labels1 = ward_linkage_labels, dbscan_labels

pd.crosstab(labels0, labels1, dropna=False)

### Interpret clusterings

In [ ]:
labels0 = ward_linkage_labels

(data.assign(**{"cluster": labels0}).groupby("cluster", observed=True).agg(["mean", "var"]))